In [1]:
import glob, time, joblib
from pathlib import Path
import numpy as np
import polars as pl
import torch

FEAT = sorted({p.rsplit("/", 1)[0] for p in glob.glob("/kaggle/input/**/train_text.parquet", recursive=True)})[0]
cols = ["complaint_id", "product", "text_tfidf"]
train = pl.read_parquet(f"{FEAT}/train_text.parquet", columns=cols)
val   = pl.read_parquet(f"{FEAT}/val_text.parquet",   columns=cols)
print(train.shape, val.shape)

CLASSES = sorted(train["product"].unique().to_list())
L2I = {c: i for i, c in enumerate(CLASSES)}
y_train = train["product"].replace_strict(L2I).to_numpy()
y_val   = val["product"].replace_strict(L2I).to_numpy()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Machine :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

CACHE = Path("/kaggle/working/cache"); CACHE.mkdir(parents=True, exist_ok=True)
REPORTS = Path("/kaggle/working/reports_l4"); REPORTS.mkdir(parents=True, exist_ok=True)

def cached(name, fn):
    """Recharge 'name' depuis working/cache ou n'importe quel input, sinon calcule."""
    candidates = [CACHE / name] + [Path(p) for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)]
    for path in candidates:
        if path.exists():
            obj = joblib.load(path)
            if path.parent != CACHE:
                joblib.dump(obj, CACHE / name)
            print(f"[cache] {name} rechargé")
            return obj
    t0 = time.time()
    obj = fn()
    joblib.dump(obj, CACHE / name)
    print(f"[cache] {name} calculé et sauvegardé ({time.time() - t0:.0f} s)")
    return obj

(100000, 3) (50000, 3)
Machine : cuda Tesla T4


In [2]:
!pip install -q gensim
from gensim.models import KeyedVectors

# 1. La carte Word2Vec de L3
W2V_PATH = glob.glob("/kaggle/input/**/w2v_complaints_d100.txt", recursive=True)[0]
w2v = KeyedVectors.load_word2vec_format(W2V_PATH)
print(f"Carte Word2Vec : {len(w2v.key_to_index):,} mots, {w2v.vector_size} nombres par mot")

# 2. Vocabulaire : 0 = <pad>, 1 = <unk>, puis les mots de la carte
ITOS = ["<pad>", "<unk>"] + list(w2v.key_to_index)
STOI = {w: i for i, w in enumerate(ITOS)}
print(f"Vocabulaire du réseau : {len(ITOS):,} numéros")

# 3. Longueur des plaintes (en mots), pour choisir où couper
lens = np.array([len(t.split()) for t in train["text_tfidf"].to_list()])
print("\nLongueur des plaintes (mots, après nettoyage) :")
for q in [50, 75, 90, 95, 99]:
    print(f"  p{q} : {np.percentile(lens, q):.0f}")
for L in [128, 200, 256, 400]:
    print(f"  coupées si longueur max = {L} : {(lens > L).mean():.1%}")

Carte Word2Vec : 15,093 mots, 100 nombres par mot
Vocabulaire du réseau : 15,095 numéros

Longueur des plaintes (mots, après nettoyage) :
  p50 : 67
  p75 : 119
  p90 : 203
  p95 : 280
  p99 : 511
  coupées si longueur max = 128 : 22.3%
  coupées si longueur max = 200 : 10.2%
  coupées si longueur max = 256 : 6.2%
  coupées si longueur max = 400 : 1.9%


In [3]:
MAX_LEN = 256          # à ajuster après la cellule 2

def encode(texts, max_len=MAX_LEN):
    """Chaque plainte devient une ligne de max_len numéros (coupée ou complétée par des 0)."""
    X = np.zeros((len(texts), max_len), dtype=np.int32)
    n_unk = n_tok = 0
    for i, t in enumerate(texts):
        ids = [STOI.get(w, 1) for w in t.split()][:max_len]
        X[i, :len(ids)] = ids
        n_unk += sum(1 for x in ids if x == 1); n_tok += len(ids)
    print(f"  {len(texts):,} plaintes, {n_unk / n_tok:.1%} de mots inconnus (<unk>)")
    return X

data = cached(f"encoded_len{MAX_LEN}.joblib", lambda: {
    "X_train": encode(train["text_tfidf"].to_list()),
    "X_val":   encode(val["text_tfidf"].to_list()),
})
X_train, X_val = data["X_train"], data["X_val"]
print("Forme du train :", X_train.shape)

# Exemple : une plainte, ses mots, ses numéros
print("\nMots    :", train["text_tfidf"][0].split()[:12])
print("Numéros :", X_train[0, :12].tolist())

# 4. Carte de départ pour l'étape 4 : une ligne de 100 nombres par numéro
EMB = np.zeros((len(ITOS), w2v.vector_size), dtype=np.float32)        # <pad> = que des 0
EMB[1] = np.random.default_rng(42).normal(0, 0.1, w2v.vector_size)    # <unk> = petit vecteur au hasard
for w, i in STOI.items():
    if i >= 2:
        EMB[i] = w2v[w]
print(f"\nMatrice d'embedding Word2Vec : {EMB.shape} (une ligne par mot du vocabulaire)")

  100,000 plaintes, 0.6% de mots inconnus (<unk>)
  50,000 plaintes, 0.6% de mots inconnus (<unk>)
[cache] encoded_len256.joblib calculé et sauvegardé (4 s)
Forme du train : (100000, 256)

Mots    : ['[DATE]', 'switch', 'phone', 'carry', '[PII]', '[PII]', '[PII]', 'pay', 'rest', 'phone', 'charge', 'leave']
Numéros : [6, 862, 48, 1379, 2, 2, 2, 12, 1161, 48, 29, 197]

Matrice d'embedding Word2Vec : (15095, 100) (une ligne par mot du vocabulaire)


In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# Mêmes poids de classe qu'en L2 et L3 (alpha = 0.75)
POOL = {"credit_reporting": 569_176, "debt_collection": 217_966, "credit_card": 121_859,
        "mortgage": 112_942, "checking_savings": 101_743, "student_loan": 39_948,
        "money_transfer": 38_190, "vehicle_loan": 33_968, "payday_personal_loan": 23_654,
        "prepaid_card": 12_417}
N_TRAIN = np.bincount(y_train, minlength=len(CLASSES))
raw = np.array([POOL[c] ** (0.75 - 0.5) for c in CLASSES])
CW = torch.tensor(raw / ((raw * N_TRAIN).sum() / N_TRAIN.sum()), dtype=torch.float32)

train_ds = TensorDataset(torch.from_numpy(X_train).long(), torch.from_numpy(y_train).long())
val_ds   = TensorDataset(torch.from_numpy(X_val).long(),   torch.from_numpy(y_val).long())

def predict(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for xb, _ in loader:
            out.append(model(xb.to(DEVICE)).argmax(1).cpu())
    return torch.cat(out).numpy()

def run_training(make_model, name, epochs=10, lr=1e-3, batch=128, patience=2, seed=42):
    """Entraîne, mesure le macro-F1 val à chaque époque, garde la meilleure version (arrêt anticipé)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"=== {name} : {n_params:,} paramètres entraînables ===")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=CW.to(DEVICE))
    tl = DataLoader(train_ds, batch_size=batch, shuffle=True, generator=torch.Generator().manual_seed(seed))
    vl = DataLoader(val_ds, batch_size=512)
    best, history, bad = {"f1": -1}, [], 0
    for ep in range(1, epochs + 1):
        t0 = time.time(); model.train(); total = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        pred = predict(model, vl)
        f1, acc = f1_score(y_val, pred, average="macro"), accuracy_score(y_val, pred)
        history.append({"époque": ep, "perte_train": round(total / len(train_ds), 4),
                        "macro_f1_val": round(f1, 4), "accuracy_val": round(acc, 4)})
        print(f"  époque {ep:2d} | perte train {total / len(train_ds):.4f} | macro-F1 val {f1:.4f} "
              f"| accuracy {acc:.4f} | {time.time() - t0:.0f} s")
        if f1 > best["f1"]:
            best = {"f1": f1, "acc": acc, "epoch": ep, "pred": pred,
                    "state": {k: v.cpu().clone() for k, v in model.state_dict().items()}}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"  arrêt : pas de progrès depuis {patience} époques")
                break
    print(f"  meilleure époque : {best['epoch']} (macro-F1 {best['f1']:.4f})\n")
    return {"best": best, "history": history, "n_params": n_params}

In [5]:
class TextCNN(nn.Module):
    """Kim (2014) : embeddings -> surligneurs de 3, 4 et 5 mots -> max pooling -> 10 scores."""
    def __init__(self, emb_matrix=None, vocab=len(ITOS), dim=100, n_filters=100,
                 sizes=(3, 4, 5), n_classes=len(CLASSES), dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        if emb_matrix is not None:                                   # départ = carte Word2Vec de L3
            self.emb.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.convs = nn.ModuleList([nn.Conv1d(dim, n_filters, k) for k in sizes])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(sizes), n_classes)

    def forward(self, x):                                            # x : (batch, 256) numéros de mots
        e = self.emb(x).transpose(1, 2)                              # (batch, 100, 256) positions sur la carte
        h = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]   # score max de chaque surligneur
        return self.fc(self.drop(torch.cat(h, dim=1)))               # (batch, 10) un score par produit

res_cnn_rand = cached("cnn_random_v1.joblib",
                      lambda: run_training(lambda: TextCNN(), "TextCNN, carte au hasard"))
res_cnn_w2v  = cached("cnn_w2v_v1.joblib",
                      lambda: run_training(lambda: TextCNN(EMB), "TextCNN, carte Word2Vec"))

# Comparaison avec les lots précédents
results = [
    {"modèle": "TF-IDF + SVM réglé (L2)",          "macro_f1": 0.763, "accuracy": 0.856},
    {"modèle": "Moyenne Word2Vec + LR (L3)",        "macro_f1": 0.700, "accuracy": 0.829},
    {"modèle": "TextCNN, carte au hasard",          "macro_f1": round(res_cnn_rand["best"]["f1"], 3),
     "accuracy": round(res_cnn_rand["best"]["acc"], 3)},
    {"modèle": "TextCNN, carte Word2Vec",           "macro_f1": round(res_cnn_w2v["best"]["f1"], 3),
     "accuracy": round(res_cnn_w2v["best"]["acc"], 3)},
]
display(pl.DataFrame(results).sort("macro_f1", descending=True))

# Courbes d'apprentissage : macro-F1 val à chaque époque
curves = pl.DataFrame([{"version": "hasard", **h} for h in res_cnn_rand["history"]] +
                      [{"version": "Word2Vec", **h} for h in res_cnn_w2v["history"]])
display(curves.pivot(on="version", index="époque", values="macro_f1_val"))
curves.write_csv(REPORTS / "cnn_learning_curves.csv")

=== TextCNN, carte au hasard : 1,632,810 paramètres entraînables ===
  époque  1 | perte train 1.0390 | macro-F1 val 0.6820 | accuracy 0.8221 | 18 s
  époque  2 | perte train 0.7709 | macro-F1 val 0.7123 | accuracy 0.8364 | 15 s
  époque  3 | perte train 0.6884 | macro-F1 val 0.7255 | accuracy 0.8385 | 15 s
  époque  4 | perte train 0.6391 | macro-F1 val 0.7272 | accuracy 0.8356 | 16 s
  époque  5 | perte train 0.6011 | macro-F1 val 0.7377 | accuracy 0.8402 | 16 s
  époque  6 | perte train 0.5704 | macro-F1 val 0.7404 | accuracy 0.8448 | 16 s
  époque  7 | perte train 0.5412 | macro-F1 val 0.7454 | accuracy 0.8489 | 15 s
  époque  8 | perte train 0.5168 | macro-F1 val 0.7386 | accuracy 0.8377 | 15 s
  époque  9 | perte train 0.4930 | macro-F1 val 0.7414 | accuracy 0.8447 | 15 s
  arrêt : pas de progrès depuis 2 époques
  meilleure époque : 7 (macro-F1 0.7454)

[cache] cnn_random_v1.joblib calculé et sauvegardé (148 s)
=== TextCNN, carte Word2Vec : 1,632,810 paramètres entraînables ===


modèle,macro_f1,accuracy
str,f64,f64
"""TF-IDF + SVM réglé (L2)""",0.763,0.856
"""TextCNN, carte Word2Vec""",0.76,0.854
"""TextCNN, carte au hasard""",0.745,0.849
"""Moyenne Word2Vec + LR (L3)""",0.7,0.829


époque,hasard,Word2Vec
i64,f64,f64
1,0.682,0.7438
2,0.7123,0.7535
3,0.7255,0.7604
4,0.7272,0.7555
5,0.7377,0.7585
6,0.7404,null
7,0.7454,null
8,0.7386,null
9,0.7414,null


In [6]:
from sklearn.metrics import classification_report

f1_cnn = f1_score(y_val, res_cnn_w2v["best"]["pred"], labels=list(range(len(CLASSES))), average=None)
L2_F1 = {"checking_savings": 0.812, "credit_card": 0.745, "credit_reporting": 0.922, "debt_collection": 0.715,
         "money_transfer": 0.706, "mortgage": 0.873, "payday_personal_loan": 0.542, "prepaid_card": 0.735,
         "student_loan": 0.868, "vehicle_loan": 0.708}
L3_MEAN_F1 = {"checking_savings": 0.771, "credit_card": 0.690, "credit_reporting": 0.910, "debt_collection": 0.678,
              "money_transfer": 0.656, "mortgage": 0.833, "payday_personal_loan": 0.404, "prepaid_card": 0.660,
              "student_loan": 0.812, "vehicle_loan": 0.610}     # moyenne FastText + LR (L3)

per_class = pl.DataFrame({
    "classe": CLASSES,
    "moyenne (L3)": [L3_MEAN_F1[c] for c in CLASSES],
    "TextCNN W2V": np.round(f1_cnn, 3),
    "SVM (L2)": [L2_F1[c] for c in CLASSES],
}).with_columns(
    (pl.col("TextCNN W2V") - pl.col("moyenne (L3)")).round(3).alias("gain vs moyenne"),
    (pl.col("TextCNN W2V") - pl.col("SVM (L2)")).round(3).alias("écart vs SVM"),
).sort("gain vs moyenne", descending=True)
display(per_class)
per_class.write_csv(REPORTS / "cnn_per_class.csv")

classe,moyenne (L3),TextCNN W2V,SVM (L2),gain vs moyenne,écart vs SVM
str,f64,f64,f64,f64,f64
"""payday_personal_loan""",0.404,0.542,0.542,0.138,0.0
"""vehicle_loan""",0.61,0.709,0.708,0.099,0.001
"""student_loan""",0.812,0.874,0.868,0.062,0.006
"""prepaid_card""",0.66,0.721,0.735,0.061,-0.014
"""mortgage""",0.833,0.888,0.873,0.055,0.015
"""credit_card""",0.69,0.735,0.745,0.045,-0.01
"""checking_savings""",0.771,0.81,0.812,0.039,-0.002
"""money_transfer""",0.656,0.694,0.706,0.038,-0.012
"""debt_collection""",0.678,0.71,0.715,0.032,-0.005


In [7]:
class BiLSTMAttention(nn.Module):
    """Embeddings -> BiLSTM (lecture dans les 2 sens) -> attention (poids par mot) -> 10 scores."""
    def __init__(self, emb_matrix=None, vocab=len(ITOS), dim=100, hidden=128,
                 n_classes=len(CLASSES), dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        if emb_matrix is not None:
            self.emb.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.lstm = nn.LSTM(dim, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Sequential(nn.Linear(2 * hidden, 64), nn.Tanh(), nn.Linear(64, 1))
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, x, return_attention=False):
        mask = x != 0                                               # True = vrai mot, False = remplissage
        lengths = mask.sum(1).clamp(min=1).cpu()
        e = self.drop(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)                                  # lecture mot à mot, sans les remplissages
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        scores = self.att(out).squeeze(-1).masked_fill(~mask, -1e9)   # importance de chaque mot
        weights = torch.softmax(scores, dim=1)                      # poids qui somment à 1, 0 sur les remplissages
        summary = (weights.unsqueeze(-1) * out).sum(dim=1)          # moyenne pondérée par l'attention
        logits = self.fc(self.drop(summary))
        return (logits, weights) if return_attention else logits

res_lstm_rand = cached("bilstm_random_v1.joblib",
                       lambda: run_training(lambda: BiLSTMAttention(), "BiLSTM + attention, carte au hasard"))
res_lstm_w2v  = cached("bilstm_w2v_v1.joblib",
                       lambda: run_training(lambda: BiLSTMAttention(EMB), "BiLSTM + attention, carte Word2Vec"))

results += [
    {"modèle": "BiLSTM + attention, carte au hasard", "macro_f1": round(res_lstm_rand["best"]["f1"], 3),
     "accuracy": round(res_lstm_rand["best"]["acc"], 3)},
    {"modèle": "BiLSTM + attention, carte Word2Vec",  "macro_f1": round(res_lstm_w2v["best"]["f1"], 3),
     "accuracy": round(res_lstm_w2v["best"]["acc"], 3)},
]
table = pl.DataFrame(results).sort("macro_f1", descending=True)
display(table)
table.write_csv(REPORTS / "l4_comparison.csv")

curves_lstm = pl.DataFrame([{"version": "hasard", **h} for h in res_lstm_rand["history"]] +
                           [{"version": "Word2Vec", **h} for h in res_lstm_w2v["history"]])
display(curves_lstm.pivot(on="version", index="époque", values="macro_f1_val"))
curves_lstm.write_csv(REPORTS / "bilstm_learning_curves.csv")

=== BiLSTM + attention, carte au hasard : 1,764,103 paramètres entraînables ===
  époque  1 | perte train 1.0106 | macro-F1 val 0.6612 | accuracy 0.8128 | 58 s
  époque  2 | perte train 0.7269 | macro-F1 val 0.6957 | accuracy 0.8263 | 58 s
  époque  3 | perte train 0.6605 | macro-F1 val 0.7218 | accuracy 0.8353 | 58 s
  époque  4 | perte train 0.6247 | macro-F1 val 0.7184 | accuracy 0.8253 | 57 s
  époque  5 | perte train 0.5979 | macro-F1 val 0.7362 | accuracy 0.8396 | 58 s
  époque  6 | perte train 0.5793 | macro-F1 val 0.7414 | accuracy 0.8487 | 58 s
  époque  7 | perte train 0.5631 | macro-F1 val 0.7378 | accuracy 0.8457 | 58 s
  époque  8 | perte train 0.5468 | macro-F1 val 0.7439 | accuracy 0.8409 | 58 s
  époque  9 | perte train 0.5350 | macro-F1 val 0.7469 | accuracy 0.8464 | 58 s
  époque 10 | perte train 0.5243 | macro-F1 val 0.7390 | accuracy 0.8385 | 58 s
  meilleure époque : 9 (macro-F1 0.7469)

[cache] bilstm_random_v1.joblib calculé et sauvegardé (577 s)
=== BiLSTM + att

modèle,macro_f1,accuracy
str,f64,f64
"""BiLSTM + attention, carte Word…",0.766,0.858
"""TF-IDF + SVM réglé (L2)""",0.763,0.856
"""TextCNN, carte Word2Vec""",0.76,0.854
"""BiLSTM + attention, carte au h…",0.747,0.846
"""TextCNN, carte au hasard""",0.745,0.849
"""Moyenne Word2Vec + LR (L3)""",0.7,0.829


époque,hasard,Word2Vec
i64,f64,f64
1,0.6612,0.7334
2,0.6957,0.7396
3,0.7218,0.7573
4,0.7184,0.7448
5,0.7362,0.7598
6,0.7414,0.7661
7,0.7378,0.7613
8,0.7439,0.7585
9,0.7469,null


In [8]:
# 1. Recharger le SVM de L2 et le faire prédire sur la même val
svm_path = glob.glob("/kaggle/input/**/svm_tfidf_v1.joblib", recursive=True)[0]
svm_bundle = joblib.load(svm_path)
X_val_tfidf = svm_bundle["vectorizer"].transform(val["text_tfidf"].to_list())
p_svm = np.array([L2I[c] for c in svm_bundle["model"].predict(X_val_tfidf)])
print(f"SVM rechargé : macro-F1 {f1_score(y_val, p_svm, average='macro'):.4f}  (attendu 0.7626)")

p_cnn  = res_cnn_w2v["best"]["pred"]
p_lstm = res_lstm_w2v["best"]["pred"]

# 2. Bootstrap apparié : 1 000 tirages des mêmes plaintes pour les deux modèles
K = len(CLASSES)
def macro_f1_fast(y, p):
    cm = np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    tp = np.diag(cm)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec, rec = np.nan_to_num(tp / cm.sum(0)), np.nan_to_num(tp / cm.sum(1))
        return np.nan_to_num(2 * prec * rec / (prec + rec)).mean()

def paired_bootstrap(pa, pb, B=1_000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_val)
    diff = np.empty(B)
    for b in range(B):
        i = rng.integers(0, n, n)
        diff[b] = macro_f1_fast(y_val[i], pa[i]) - macro_f1_fast(y_val[i], pb[i])
    return diff

rows = []
for name_a, pa, name_b, pb in [("BiLSTM W2V", p_lstm, "SVM (L2)", p_svm),
                               ("TextCNN W2V", p_cnn, "SVM (L2)", p_svm),
                               ("BiLSTM W2V", p_lstm, "TextCNN W2V", p_cnn)]:
    d = paired_bootstrap(pa, pb)
    rows.append({"comparaison": f"{name_a} - {name_b}", "écart": round(d.mean(), 4),
                 "IC 95 % bas": round(np.percentile(d, 2.5), 4), "IC 95 % haut": round(np.percentile(d, 97.5), 4),
                 "part où A ne gagne pas": round((d <= 0).mean(), 3)})
boot = pl.DataFrame(rows)
display(boot)
boot.write_csv(REPORTS / "bootstrap_l4_vs_l2.csv")

# 3. Bonus : les modèles se trompent-ils sur les mêmes plaintes ?
both_wrong = ((p_lstm != y_val) & (p_svm != y_val)).mean()
only_svm   = ((p_lstm == y_val) & (p_svm != y_val)).mean()
only_lstm  = ((p_lstm != y_val) & (p_svm == y_val)).mean()
print(f"\nErreurs communes BiLSTM et SVM : {both_wrong:.1%} des plaintes")
print(f"Seul le SVM se trompe          : {only_svm:.1%}")
print(f"Seul le BiLSTM se trompe       : {only_lstm:.1%}")

SVM rechargé : macro-F1 0.7626  (attendu 0.7626)


comparaison,écart,IC 95 % bas,IC 95 % haut,part où A ne gagne pas
str,f64,f64,f64,f64
"""BiLSTM W2V - SVM (L2)""",0.0036,-0.0017,0.0093,0.089
"""TextCNN W2V - SVM (L2)""",-0.0021,-0.0074,0.0032,0.798
"""BiLSTM W2V - TextCNN W2V""",0.0058,0.0004,0.0112,0.016



Erreurs communes BiLSTM et SVM : 11.4% des plaintes
Seul le SVM se trompe          : 3.0%
Seul le BiLSTM se trompe       : 2.8%


In [9]:
import html
from IPython.display import HTML, display as show

model = BiLSTMAttention().to(DEVICE)
model.load_state_dict(res_lstm_w2v["best"]["state"])
model.eval()

def attention_of(indices):
    xb = torch.from_numpy(X_val[indices]).long().to(DEVICE)
    with torch.no_grad():
        logits, w = model(xb, return_attention=True)
    return logits.argmax(1).cpu().numpy(), w.cpu().numpy()

def render(i, pred, w):
    ids = X_val[i]
    n = int((ids != 0).sum())
    a = w[:n] / w[:n].max()
    spans = "".join(f'<span style="background:rgba(214,96,77,{x:.2f});padding:1px 3px;margin:1px;'
                    f'border-radius:3px">{html.escape(ITOS[t])}</span> ' for t, x in zip(ids[:n], a))
    ok = "correct" if pred == y_val[i] else "ERREUR"
    return (f"<p style='margin:14px 0 4px'><b>Vraie classe : {CLASSES[y_val[i]]} | Prédite : {CLASSES[pred]} ({ok})</b></p>"
            f"<p style='line-height:2;font-family:monospace;font-size:13px'>{spans}</p>")

# Exemples : une plainte bien classée par produit (30 à 70 mots), puis 3 erreurs
lens_val = (X_val != 0).sum(1)
rng = np.random.default_rng(0)
picks = []
for c in range(len(CLASSES)):
    cand = np.where((y_val == c) & (p_lstm == c) & (lens_val >= 30) & (lens_val <= 70))[0]
    picks.append(int(rng.choice(cand)))
errs = np.where((p_lstm != y_val) & (lens_val >= 30) & (lens_val <= 70))[0]
picks += [int(x) for x in rng.choice(errs, 3, replace=False)]

preds_ex, att_ex = attention_of(picks)
page = "".join(render(i, p, w) for i, p, w in zip(picks, preds_ex, att_ex))
show(HTML(page))
(REPORTS / "attention_examples.html").write_text(f"<html><meta charset='utf-8'><body style='max-width:900px'>{page}</body></html>")

68571

In [10]:
V = len(ITOS)
att_sum = np.zeros((len(CLASSES), V))
att_cnt = np.zeros((len(CLASSES), V))
loader = DataLoader(val_ds, batch_size=512)
with torch.no_grad():
    for xb, _ in loader:
        logits, w = model(xb.to(DEVICE), return_attention=True)
        pred = logits.argmax(1).cpu().numpy()
        w = w.cpu().numpy()
        x = xb.numpy()
        mask = x != 0
        rel = w * mask.sum(1, keepdims=True)              # attention relative : 1 = moyenne de la plainte
        rows_c = np.repeat(pred[:, None], x.shape[1], axis=1)[mask]
        np.add.at(att_sum, (rows_c, x[mask]), rel[mask])
        np.add.at(att_cnt, (rows_c, x[mask]), 1)

with np.errstate(divide="ignore", invalid="ignore"):
    att_mean = np.where(att_cnt >= 30, att_sum / att_cnt, 0)   # au moins 30 occurrences pour être fiable

rows = []
for c, name in enumerate(CLASSES):
    top = np.argsort(att_mean[c])[::-1][:12]
    rows.append({"produit prédit": name,
                 "mots les plus regardés (attention relative)": ", ".join(f"{ITOS[t]} ({att_mean[c, t]:.1f})" for t in top)})
att_top = pl.DataFrame(rows)
with pl.Config(fmt_str_lengths=300, tbl_width_chars=300):
    display(att_top)
att_top.write_csv(REPORTS / "attention_top_words.csv")

produit prédit,mots les plus regardés (attention relative)
str,str
"""checking_savings""","""mortgage (12.9), provisional (12.2), webster (11.1), student (10.6), chime (10.3), serve (10.2), santander (10.2), robinhood (10.1), ally (9.9), goldman (9.4), main (9.3), loan (9.3)"""
"""credit_card""","""paypal (16.0), affirm (15.4), carecredit (15.3), kohl (13.5), fortiva (13.1), elan (13.0), sapphire (12.9), venmo (12.8), ally (12.6), apple (12.4), barclay (11.6), comenity (11.2)"""
"""credit_reporting""","""certifiable (27.9), certifiably (22.0), equifax (21.4), 1681e (21.2), stickler (20.5), innovis (20.5), experian (20.3), transunion (20.0), lexisnexis (20.0), santander (19.9), trans (19.8), checkr (19.5)"""
"""debt_collection""","""credence (17.4), ncb (17.3), acceptance (16.9), procollect (16.3), sunrise (15.7), radius (15.0), hunter (14.7), caine (14.6), iq (14.4), wakefield (14.4), jefferson (14.1), midland (13.3)"""
"""money_transfer""","""gemini (15.5), crypto.com (15.0), remitly (14.3), western (14.2), square (13.2), coinbase (12.5), paypal (11.2), cashapp (11.0), venmo (10.6), bitcoin (10.3), zelle (10.2), international (9.9)"""
"""mortgage""","""loancare (22.0), bsi (19.0), heloc (16.9), selene (16.7), cooper (16.3), lakeview (15.6), pennymac (15.4), specialized (14.4), conventional (14.1), fay (13.9), foreclose (13.7), shellpoint (13.6)"""
"""payday_personal_loan""","""klarna (17.3), goodleap (16.9), payday (16.2), main (15.9), mosaic (15.5), onemain (14.9), netcredit (13.0), affirm (12.4), solar (10.7), finance (7.1), loan (7.0), line (6.2)"""
"""prepaid_card""","""prepaid (24.2), prepay (19.8), vanilla (19.7), unemployment (18.0), incomm (17.5), netspend (17.1), direct (14.0), network (13.3), gift (11.5), load (8.1), benefit (5.1), amex (4.7)"""
"""student_loan""","""edfinancial (17.5), firstmark (16.9), aidvantage (16.8), nelnet (15.9), mohela (15.8), studentaid.gov (13.3), navient (12.4), pslf (10.6), ibr (10.6), ed (9.3), sallie (9.1), idr (8.5)"""


In [11]:
!pip install -q vaderSentiment
split_val = [p for p in glob.glob("/kaggle/input/**/val.parquet", recursive=True) if "_text" not in p][0]
meta = pl.read_parquet(split_val, columns=["complaint_id", "company_response_to_consumer", "timely_response", "company"])
txt = pl.read_parquet(f"{FEAT}/val_text.parquet", columns=["complaint_id", "product", "text_light"])
sent = txt.join(meta, on="complaint_id", how="left").sample(20_000, seed=42)
print(sent.shape)
display(sent.group_by("company_response_to_consumer").len().sort("len", descending=True))
display(sent.group_by("timely_response").len())

(20000, 6)


company_response_to_consumer,len
str,u32
"""Closed with explanation""",12876
"""Closed with non-monetary relie…",6422
"""Closed with monetary relief""",671
"""Untimely response""",31


timely_response,len
str,u32
"""Yes""",19829
"""No""",171


In [12]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def run_vader():
    sia = SentimentIntensityAnalyzer()
    return np.array([sia.polarity_scores(t)["compound"] for t in sent["text_light"].to_list()])

vader = cached("sent_vader_val20k.joblib", run_vader)
print(f"VADER : score moyen {vader.mean():+.3f} | négatives (< -0.05) : {(vader < -0.05).mean():.1%} "
      f"| positives (> 0.05) : {(vader > 0.05).mean():.1%}")

[cache] sent_vader_val20k.joblib calculé et sauvegardé (60 s)
VADER : score moyen -0.008 | négatives (< -0.05) : 47.5% | positives (> 0.05) : 48.0%


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

def run_roberta():
    tok = AutoTokenizer.from_pretrained(SENT_MODEL)
    m = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL).to(DEVICE).half().eval()
    labels = [m.config.id2label[i].lower() for i in range(m.config.num_labels)]
    texts = sent["text_light"].to_list()
    order = np.argsort([len(t) for t in texts])                 # paquets de longueurs proches = plus rapide
    probs = np.zeros((len(texts), len(labels)), dtype=np.float32)
    for start in range(0, len(texts), 64):
        idx = order[start:start + 64]
        enc = tok([texts[i] for i in idx], truncation=True, max_length=512,
                  padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            probs[idx] = torch.softmax(m(**enc).logits.float(), dim=-1).cpu().numpy()
    return {"labels": labels, "probs": probs}

rob_out = cached("sent_roberta_val20k.joblib", run_roberta)
labels, probs = rob_out["labels"], rob_out["probs"]
print("Étiquettes du modèle :", labels)
roberta = probs[:, labels.index("positive")] - probs[:, labels.index("negative")]
rob_label = np.array(labels)[probs.argmax(1)]
print(f"RoBERTa : score moyen {roberta.mean():+.3f} | "
      + " | ".join(f"{l} : {(rob_label == l).mean():.1%}" for l in labels))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

[cache] sent_roberta_val20k.joblib calculé et sauvegardé (91 s)
Étiquettes du modèle : ['negative', 'neutral', 'positive']
RoBERTa : score moyen -0.559 | negative : 69.1% | neutral : 30.7% | positive : 0.1%


In [14]:
from scipy.stats import spearmanr

sent = sent.with_columns(pl.Series("vader", vader), pl.Series("roberta", roberta))

# 1. Accord entre les deux méthodes
rho, _ = spearmanr(vader, roberta)
print(f"Corrélation VADER / RoBERTa (Spearman) : {rho:.3f}")

# 2. Ton moyen selon la réponse de l'entreprise
by_resp = (sent.group_by("company_response_to_consumer")
               .agg(pl.len().alias("n"), pl.col("vader").mean().round(3), pl.col("roberta").mean().round(3))
               .filter(pl.col("n") >= 100).sort("roberta"))
display(by_resp)

# 3. Ton moyen selon le respect des délais
display(sent.group_by("timely_response")
            .agg(pl.len().alias("n"), pl.col("vader").mean().round(3), pl.col("roberta").mean().round(3)))

# 4. Taux de remboursement selon le ton (5 groupes, du plus négatif au moins négatif selon RoBERTa)
sent = sent.with_columns(
    pl.col("roberta").qcut(5, labels=["1 très négatif", "2", "3", "4", "5 le moins négatif"]).alias("quintile_ton"),
    (pl.col("company_response_to_consumer") == "Closed with monetary relief").alias("rembourse"))
by_q = (sent.group_by("quintile_ton")
            .agg(pl.len().alias("n"), pl.col("roberta").mean().round(3),
                 (pl.col("rembourse").mean() * 100).round(1).alias("% remboursé"))
            .sort("quintile_ton"))
display(by_q)

# 5. Même analyse par produit : le lien tient-il à l'intérieur de chaque produit ?
by_prod = (sent.group_by("product", "quintile_ton")
               .agg((pl.col("rembourse").mean() * 100).round(1).alias("% remboursé"))
               .pivot(on="quintile_ton", index="product", values="% remboursé", sort_columns=True))
display(by_prod)

# 6. Les désaccords : VADER positif, RoBERTa négatif
disagree = sent.filter((pl.col("vader") > 0.5) & (pl.col("roberta") < -0.5)).head(3)
for r in disagree.iter_rows(named=True):
    print(f"\n[VADER {r['vader']:+.2f} | RoBERTa {r['roberta']:+.2f}] {r['text_light'][:400]}")

by_resp.write_csv(REPORTS / "sentiment_by_response.csv")
by_q.write_csv(REPORTS / "sentiment_quintiles_relief.csv")
by_prod.write_csv(REPORTS / "sentiment_quintiles_relief_by_product.csv")

Corrélation VADER / RoBERTa (Spearman) : 0.215


company_response_to_consumer,n,vader,roberta
str,u32,f64,f32
"""Closed with non-monetary relie…",6422,0.032,-0.571
"""Closed with explanation""",12876,-0.018,-0.557
"""Closed with monetary relief""",671,-0.173,-0.496


timely_response,n,vader,roberta
str,u32,f64,f32
"""No""",171,-0.175,-0.556
"""Yes""",19829,-0.006,-0.559


quintile_ton,n,roberta,% remboursé
cat,u32,f32,f64
"""1 très négatif""",4000,-0.853,2.6
"""2""",4000,-0.75,3.0
"""3""",4000,-0.632,3.0
"""4""",4000,-0.454,3.5
"""5 le moins négatif""",4000,-0.108,4.7


product,1 très négatif,2,3,4,5 le moins négatif
str,f64,f64,f64,f64,f64
"""credit_card""",13.2,20.6,18.7,19.6,20.0
"""student_loan""",0.0,0.0,0.0,2.3,0.0
"""debt_collection""",0.2,0.4,0.5,0.4,0.9
"""prepaid_card""",18.2,31.0,27.0,18.5,19.4
"""money_transfer""",6.0,9.3,9.1,11.4,7.5
"""checking_savings""",14.1,14.5,11.6,15.7,17.6
"""mortgage""",2.6,2.8,0.9,3.2,2.0
"""vehicle_loan""",2.7,1.5,3.8,1.4,3.2
"""credit_reporting""",0.2,0.1,0.1,0.0,0.2



[VADER +0.66 | RoBERTa -0.83] I have a [PII] with US Bank. I have been attempting to pay off the car loan since [DATE] . They have kicked the information back for the pay off 3 times and then told me that the account I am paying the payments with is non existant. which is the account I have attempted to pay off the car loan with. When I asked them about the interested that they continued to charge me they basically told me tha

[VADER +0.74 | RoBERTa -0.64] I am filing this complaint to request the removal of inaccurate late payments and correction of non-reporting issues on my credit account ( # [PII] ) with [PII] , as reflected on my credit reports from [PII] , [PII] , and Equifax. Late payments are inaccurately reported for [PII] , [PII] , [PII] , and [DATE] , despite these payments being made on time, as evidenced by my attached payment records. 

[VADER +0.72 | RoBERTa -0.82] On [DATE] , I applied for the APPLE CARD through Goldman Sachs. I received a decline stating, " You have 

In [15]:
import re

ASPECTS = {
        "frais":          r"\b(fees?|overdraft|interest|penalt(y|ies)|apr|late charges?|service charges?)\b"
                      r"|\bcharged\b(?!\s*-?\s*off)",
    "service client": r"\b(representatives?|reps?|agents?|supervisors?|customer service|on hold|rude|hung up|"
                      r"transferred|call center|manager)\b",
    "délais":         r"\b(weeks?|months?|waiting|waited|delay(ed|s)?|no response|never received|still (have|has|not|no)|"
                      r"takes? (so )?long|years?)\b",
    "fraude":         r"\b(fraud(ulent)?|unauthori[sz]ed|scam(med|mer)?|identity theft|stolen|hacked|"
                      r"didn'?t authori[sz]e)\b",
}
ASPECT_RE = {a: re.compile(p, re.IGNORECASE) for a, p in ASPECTS.items()}
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

# 1-2. Découper en phrases et repérer les aspects
records = []                                    # (index plainte, aspect, phrase)
for i, text in enumerate(sent["text_light"].to_list()):
    for s in SENT_SPLIT.split(text):
        if len(s) < 15:
            continue
        for a, rx in ASPECT_RE.items():
            if rx.search(s):
                records.append((i, a, s[:512]))
print(f"{len(records):,} couples (phrase, aspect) trouvés dans {sent.height:,} plaintes")

# 3. Noter chaque phrase avec RoBERTa (les phrases uniques seulement)
def run_absa():
    tok = AutoTokenizer.from_pretrained(SENT_MODEL)
    m = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL).to(DEVICE).half().eval()
    labs = [m.config.id2label[i].lower() for i in range(m.config.num_labels)]
    uniq = sorted({s for _, _, s in records}, key=len)
    scores = {}
    for start in range(0, len(uniq), 128):
        batch = uniq[start:start + 128]
        enc = tok(batch, truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(m(**enc).logits.float(), dim=-1).cpu().numpy()
        for s, row in zip(batch, p):
            scores[s] = float(row[labs.index("positive")] - row[labs.index("negative")])
    return scores

sent_scores = cached("absa_sentence_scores_v2.joblib", run_absa)
absa = pl.DataFrame({"idx": [r[0] for r in records], "aspect": [r[1] for r in records],
                     "phrase": [r[2] for r in records]}) \
         .with_columns(pl.col("phrase").replace_strict(sent_scores, return_dtype=pl.Float64).alias("score"))

# Vérification à l'œil : 2 phrases par aspect
for a in ASPECTS:
    print(f"\n=== {a} ===")
    for r in absa.filter(pl.col("aspect") == a).sample(2, seed=3).iter_rows(named=True):
        print(f"  [{r['score']:+.2f}] {r['phrase'][:200]}")

41,186 couples (phrase, aspect) trouvés dans 20,000 plaintes


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] absa_sentence_scores_v2.joblib calculé et sauvegardé (25 s)

=== frais ===
  [-0.72] Failure to comply within the stipulated [PII] timeframe will be viewed as willful noncompliance, subject to penalties under applicable laws, including identity theft laws.
  [-0.09] ( [PII] ) [PII] references ( [PII] ) Penalties for disclosure of information by preparers of returns for penalty for disclosure or use of information by preparers of returns, see section [PII] .

=== service client ===
  [-0.91] Their customer service phone help can not and/or does not know how to answer escrow questions and when they claim to transfer your call to another dept the calls get disconnected or falls into same he
  [-0.86] Meanwhile we call customer service several times, leave our number, never get through to anyone and never receive a call back.

=== délais ===
  [-0.89] And also I can not keep waiting as this investigation has taken over a year, which is insane.
  [-0.87] I have called their fraud li

In [16]:
# Un score par (plainte, aspect) = moyenne de ses phrases
per_doc = (absa.group_by("idx", "aspect").agg(pl.col("score").mean())
               .join(sent.with_row_index("idx").select("idx", "product", "rembourse")
                         .with_columns(pl.col("idx").cast(pl.Int64)),
                     on="idx", how="left"))
N = sent.height

# 1. De quoi se plaint-on ? (% des plaintes qui mentionnent l'aspect, par produit)
n_prod = dict(sent.group_by("product").len().iter_rows())
freq = (per_doc.group_by("product", "aspect").len()
               .with_columns((pl.col("len") / pl.col("product").replace_strict(n_prod) * 100).round(1).alias("%"))
               .pivot(on="aspect", index="product", values="%", sort_columns=True).sort("product"))
print("% des plaintes qui mentionnent chaque aspect, par produit :")
display(freq)

# 2. Quel aspect est le plus mal vécu ?
overall = (per_doc.group_by("aspect")
                  .agg(pl.len().alias("plaintes"), (pl.len() / N * 100).round(1).alias("% des plaintes"),
                       pl.col("score").mean().round(3).alias("sentiment moyen"))
                  .sort("sentiment moyen"))
display(overall)

# 3. Remboursement selon l'aspect mentionné
base = sent["rembourse"].mean() * 100
relief = (per_doc.group_by("aspect")
                 .agg((pl.col("rembourse").mean() * 100).round(1).alias("% remboursé si mentionné"))
                 .with_columns(pl.lit(round(base, 1)).alias("% remboursé global"))
                 .sort("% remboursé si mentionné", descending=True))
display(relief)

freq.write_csv(REPORTS / "absa_frequency_by_product.csv")
overall.write_csv(REPORTS / "absa_sentiment_by_aspect.csv")
relief.write_csv(REPORTS / "absa_relief_by_aspect.csv")

% des plaintes qui mentionnent chaque aspect, par produit :


product,délais,frais,fraude,service client
str,f64,f64,f64,f64
"""checking_savings""",55.6,21.4,39.3,37.6
"""credit_card""",53.9,35.1,31.0,34.3
"""credit_reporting""",30.2,5.6,35.9,6.4
"""debt_collection""",34.6,11.5,22.9,14.1
"""money_transfer""",54.5,14.8,42.8,37.9
"""mortgage""",71.9,37.7,11.3,40.6
"""payday_personal_loan""",57.1,34.9,17.5,25.0
"""prepaid_card""",48.8,16.0,32.7,34.6
"""student_loan""",65.3,30.4,8.9,34.0


aspect,plaintes,% des plaintes,sentiment moyen
str,u32,f64,f64
"""fraude""",6498,32.5,-0.66
"""frais""",2524,12.6,-0.436
"""délais""",7650,38.2,-0.422
"""service client""",3078,15.4,-0.417


aspect,% remboursé si mentionné,% remboursé global
str,f64,f64
"""frais""",11.1,3.4
"""service client""",9.6,3.4
"""délais""",5.3,3.4
"""fraude""",3.7,3.4


In [17]:
import statsmodels.formula.api as smf
# Une ligne par plainte : 1 si l'aspect est mentionné, 0 sinon
flags = (per_doc.select("idx", "aspect").with_columns(pl.lit(1).alias("v"))
                .pivot(on="aspect", index="idx", values="v").fill_null(0))
docs = (sent.with_row_index("idx").with_columns(pl.col("idx").cast(pl.Int64))
            .select("idx", "product", "rembourse")
            .join(flags, on="idx", how="left").fill_null(0)
            .rename({"service client": "service_client", "délais": "delais"}))
d = docs.to_pandas()
d["rembourse"] = d["rembourse"].astype(int)

rows = []
for formula, label in [("rembourse ~ frais + service_client + delais + fraude", "sans le produit"),
                       ("rembourse ~ frais + service_client + delais + fraude + C(product)", "à produit égal")]:
    m = smf.logit(formula, data=d).fit(disp=0)
    ci = m.conf_int()
    for a in ["frais", "service_client", "delais", "fraude"]:
        rows.append({"aspect": a, "modèle": label,
                     "odds ratio": round(np.exp(m.params[a]), 2),
                     "IC 95 %": f"[{np.exp(ci.loc[a, 0]):.2f} ; {np.exp(ci.loc[a, 1]):.2f}]",
                     "p-valeur": round(m.pvalues[a], 4)})
aspect_logit = pl.DataFrame(rows).sort("aspect", "modèle")
display(aspect_logit)
aspect_logit.write_csv(REPORTS / "absa_logit_relief.csv")

aspect,modèle,odds ratio,IC 95 %,p-valeur
str,str,f64,str,f64
"""delais""","""sans le produit""",1.63,"""[1.38 ; 1.93]""",0.0
"""delais""","""à produit égal""",1.18,"""[0.99 ; 1.40]""",0.0658
"""frais""","""sans le produit""",3.8,"""[3.21 ; 4.50]""",0.0
"""frais""","""à produit égal""",2.25,"""[1.88 ; 2.69]""",0.0
"""fraude""","""sans le produit""",1.24,"""[1.05 ; 1.46]""",0.0102
"""fraude""","""à produit égal""",1.14,"""[0.96 ; 1.36]""",0.1411
"""service_client""","""sans le produit""",3.1,"""[2.62 ; 3.67]""",0.0
"""service_client""","""à produit égal""",1.5,"""[1.26 ; 1.78]""",0.0
